In [ ]:
!pip install -q accelerate lpips

In [ ]:
%%writefile /kaggle/working/train_ddp_accelerate.py

# =========================================================================
# DARE-TIES (e TIES) 
# =========================================================================
# Vengono combinate in un solo modello le conoscenze specializzate dei tre 
# fine-tuning (star removal, image restoration, super-resolution), cercando 
# il modo migliore di bilanciarle
#
# Impostando dare_drop_rate_values = [0.0] si ottiene il merging TIES puro:
# si applica solo l'elezione del segno dominante per parametro e la somma 
# dei task vector concordi. DARE aggiunge uno step preliminare che azzera 
# casualmente una frazione dei parametri di ogni task vector, riducendo le 
# interferenze tra i task prima ancora che TIES risolva i conflitti di segno

import os
import math
import glob
import json
import random
import itertools
from collections import OrderedDict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from accelerate import Accelerator
from accelerate.utils import set_seed
import lpips

lpips_metric = lpips.LPIPS(net='alex')

# ==================================================================================
# ARCHITETTURA (SelfAttention, UNetBlock, SinusoidalPositionEmbeddings, ImprovedUNet)
# ==================================================================================
# una U-Net condizionale (ImprovedUNet) con blocchi residuali, embedding temporale 
# sinusoidale e self-attention. Prende in input l'immagine rumorosa x_t, il timestep 
# t e l'immagine degradata concatenata come condizione

class SelfAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.channels = channels
        self.mha = nn.MultiheadAttention(embed_dim=channels, num_heads=2, batch_first=True)
        self.ln = nn.LayerNorm(channels)
        self.ff_self = nn.Sequential(
            nn.LayerNorm(channels),
            nn.Linear(channels, channels),
            nn.GELU(),
            nn.Linear(channels, channels),
        )

    def forward(self, x):
        size = x.shape[-1]
        x_flat = x.reshape(-1, self.channels, size * size).transpose(1, 2)
        x_ln = self.ln(x_flat)
        attention_value, _ = self.mha(x_ln, x_ln, x_ln)
        attention_value = attention_value + x_flat
        attention_value = self.ff_self(attention_value) + attention_value
        return attention_value.transpose(1, 2).reshape(-1, self.channels, size, size)


class UNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim=32, num_groups=4):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.gn1 = nn.GroupNorm(num_groups, out_channels)
        self.act1 = nn.SiLU()

        self.time_mlp = nn.Linear(time_dim, out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.gn2 = nn.GroupNorm(num_groups, out_channels)
        self.act2 = nn.SiLU()

        self.residual_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x, t_emb):
        h = self.act1(self.gn1(self.conv1(x)))
        time_proj = self.time_mlp(t_emb).unsqueeze(-1).unsqueeze(-1)
        h = h + time_proj
        h = self.act2(self.gn2(self.conv2(h)))
        return h + self.residual_conv(x)


class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings


class ImprovedUNet(nn.Module):
    def __init__(self, in_channels=3, cond_channels=3, base_channels=64):
        super().__init__()
        time_dim = base_channels * 4
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(base_channels),
            nn.Linear(base_channels, time_dim),
            nn.GELU(),
            nn.Linear(time_dim, time_dim)
        )
        c = base_channels
        self.inc = UNetBlock(in_channels + cond_channels, c, time_dim)
        self.down1 = nn.Conv2d(c, c, kernel_size=4, stride=2, padding=1)
        self.enc1 = UNetBlock(c, c * 2, time_dim)
        self.down2 = nn.Conv2d(c * 2, c * 2, kernel_size=4, stride=2, padding=1)
        self.enc2 = UNetBlock(c * 2, c * 4, time_dim)
        self.down3 = nn.Conv2d(c * 4, c * 4, kernel_size=4, stride=2, padding=1)
        self.enc3 = UNetBlock(c * 4, c * 8, time_dim)
        self.attn3 = SelfAttention(c * 8)
        self.down4 = nn.Conv2d(c * 8, c * 8, kernel_size=4, stride=2, padding=1)
        self.bottleneck1 = UNetBlock(c * 8, c * 8, time_dim)
        self.attention = SelfAttention(c * 8)
        self.bottleneck2 = UNetBlock(c * 8, c * 8, time_dim)
        self.up1 = nn.ConvTranspose2d(c * 8, c * 8, kernel_size=4, stride=2, padding=1)
        self.dec1 = UNetBlock(c * 16, c * 4, time_dim)
        self.attn_up1 = SelfAttention(c * 4)
        self.up2 = nn.ConvTranspose2d(c * 4, c * 4, kernel_size=4, stride=2, padding=1)
        self.dec2 = UNetBlock(c * 8, c * 2, time_dim)
        self.up3 = nn.ConvTranspose2d(c * 2, c * 2, kernel_size=4, stride=2, padding=1)
        self.dec3 = UNetBlock(c * 4, c, time_dim)
        self.up4 = nn.ConvTranspose2d(c, c, kernel_size=4, stride=2, padding=1)
        self.dec4 = UNetBlock(c * 2, c, time_dim)
        self.out = nn.Conv2d(c, in_channels, kernel_size=3, padding=1)

    def forward(self, x_t, t, condition):
        x_input = torch.cat([x_t, condition], dim=1)
        t_emb = self.time_mlp(t.float())
        s1 = self.inc(x_input, t_emb)
        h = self.down1(s1)
        s2 = self.enc1(h, t_emb)
        h = self.down2(s2)
        s3 = self.enc2(h, t_emb)
        h = self.down3(s3)
        s4 = self.enc3(h, t_emb)
        s4 = self.attn3(s4)
        h = self.down4(s4)
        h = self.bottleneck1(h, t_emb)
        h = self.attention(h)
        h = self.bottleneck2(h, t_emb)
        h = self.up1(h)
        h = torch.cat([h, s4], dim=1)
        h = self.dec1(h, t_emb)
        h = self.attn_up1(h)
        h = self.up2(h)
        h = torch.cat([h, s3], dim=1)
        h = self.dec2(h, t_emb)
        h = self.up3(h)
        h = torch.cat([h, s2], dim=1)
        h = self.dec3(h, t_emb)
        h = self.up4(h)
        h = torch.cat([h, s1], dim=1)
        h = self.dec4(h, t_emb)
        return self.out(h)

# ==================================================================================
# DIFFUSION
# ==================================================================================
# DDPMvPrediction implementa il processo di diffusione con schedule coseno per i beta. 
# Invece di predire il rumore predice la quantità v (ovvero la combinazione di 
# rumore e immagine pulita)


def cosine_beta_schedule(timesteps, s=0.008):
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps)
    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 1e-4, 0.9999)


class DDPMvPrediction(nn.Module):
    def __init__(self, network, n_steps=300, beta_start=1e-4, beta_end=0.02):
        super().__init__()
        self.network = network
        self.n_steps = n_steps
        beta = cosine_beta_schedule(n_steps)
        alpha = 1.0 - beta
        alpha_bar = torch.cumprod(alpha, dim=0)
        alpha_bar_prev = torch.cat([torch.tensor([1.0]), alpha_bar[:-1]])
        sqrt_alpha_bar = torch.sqrt(alpha_bar)
        sqrt_one_minus_alpha_bar = torch.sqrt(1.0 - alpha_bar)

        self.register_buffer('alpha_bar', alpha_bar)
        self.register_buffer('alpha_bar_prev', alpha_bar_prev)
        self.register_buffer('sqrt_alpha_bar', sqrt_alpha_bar)
        self.register_buffer('sqrt_one_minus_alpha_bar', sqrt_one_minus_alpha_bar)
        self.register_buffer('beta', beta)
        self.register_buffer('alpha', alpha)
        self.register_buffer('posterior_variance', beta * (1 - alpha_bar_prev) / (1 - alpha_bar))

    def forward_diffusion(self, x_0, t, noise):
        sqrt_alpha_bar_t = self.sqrt_alpha_bar[t].view(-1, 1, 1, 1)
        sqrt_one_minus_alpha_bar_t = self.sqrt_one_minus_alpha_bar[t].view(-1, 1, 1, 1)
        return sqrt_alpha_bar_t * x_0 + sqrt_one_minus_alpha_bar_t * noise

    def forward(self, x_0, condition):
        batch_size = x_0.shape[0]
        t = torch.randint(0, self.n_steps, (batch_size,), device=x_0.device)
        noise = torch.randn_like(x_0)
        x_t = self.forward_diffusion(x_0, t, noise)
        sqrt_alpha_bar_t = self.sqrt_alpha_bar[t].view(-1, 1, 1, 1)
        sqrt_one_minus_alpha_bar_t = self.sqrt_one_minus_alpha_bar[t].view(-1, 1, 1, 1)
        v_target = sqrt_alpha_bar_t * noise - sqrt_one_minus_alpha_bar_t * x_0
        predicted_v = self.network(x_t, t.float(), condition)
        loss = F.mse_loss(predicted_v, v_target)
        return loss

    @torch.no_grad()
    def sample_ddim(self, condition, ddim_steps=10, eta=0.0):
        self.network.eval()
        device = self.beta.device
        batch_size, channels, h, w = condition.shape
        x = torch.randn(batch_size, channels, h, w, device=device)

        step_indices = torch.linspace(0, self.n_steps - 1, ddim_steps, dtype=torch.long)
        step_indices = torch.unique(step_indices).flip(0)

        for idx in range(len(step_indices)):
            i = step_indices[idx].item()
            t = torch.full((batch_size,), i, dtype=torch.long, device=device)
            predicted_v = self.network(x, t.float(), condition)

            alpha_bar_t = self.alpha_bar[i]
            sqrt_alpha_bar_t = self.sqrt_alpha_bar[i]
            sqrt_one_minus_alpha_bar_t = self.sqrt_one_minus_alpha_bar[i]

            pred_x0 = sqrt_alpha_bar_t * x - sqrt_one_minus_alpha_bar_t * predicted_v
            pred_x0 = torch.clamp(pred_x0, -1.0, 1.0)

            if idx == len(step_indices) - 1:
                x = pred_x0
                break

            i_prev = step_indices[idx + 1].item()
            alpha_bar_prev = self.alpha_bar[i_prev]
            pred_noise = (x - sqrt_alpha_bar_t * pred_x0) / (sqrt_one_minus_alpha_bar_t + 1e-8)

            if eta > 0.0:
                sigma_t = eta * torch.sqrt((1 - alpha_bar_prev) / (1 - alpha_bar_t) * (1 - alpha_bar_t / alpha_bar_prev))
                noise = torch.randn_like(x)
            else:
                sigma_t = torch.tensor(0.0, device=device)
                noise = torch.zeros_like(x)

            dir_xt = torch.sqrt(torch.clamp(1 - alpha_bar_prev - sigma_t ** 2, min=0.0)) * pred_noise
            x = torch.sqrt(alpha_bar_prev) * pred_x0 + dir_xt + sigma_t * noise

        self.network.train()
        return x


# =========================================================================
# CONFIG
# =========================================================================

CONFIG = {
    "theta0_path": "/kaggle/input/datasets/ilariadarchivio/theta0-pretrain/theta0.pth",
    "ckpt_key": "ema_model",
    "checkpoints": {
        "star_removal":      "/kaggle/input/datasets/ilariadarchivio/epoch-100/ddpmvpred_epoch_100.pth",
        "image_restoration": "/kaggle/input/datasets/ilariadarchivio/epoch-100/ddpmvpred_epoch_100_IR.pth",
        "super_resolution":  "/kaggle/input/datasets/ilariadarchivio/epoch-100/ddpmvpred_epoch_100_SR.pth",
    },
    "in_channels": 3,
    "cond_channels": 3,
    "base_channels": 64,
    "n_steps": 200,

    "dataset_base": "/kaggle/input/datasets/phantasm04/merged/dataset_merged",
    "stage1_folder_to_task": {
        "SR": "star_removal",
        "IR": "image_restoration",
        "SU": "super_resolution",
    },
    "stage2_folders": ["SR_IR", "SR_SU", "IR_SU", "SR_IR_SU"],

    "stage1_subsample_pct": 0.05,
    "stage2_subsample_pct": 0.05,

    "stage1_safe_zone_size": 10,
    "lambda_grid_values": {
        "star_removal":      [0.9],
        "image_restoration": [0.05, 0.15, 0.3],
        "super_resolution":  [0.3, 0.5],
    },

    # --- Parametri specifici DARE-TIES ---
    "dare_drop_rate_values": [0.5],
    "dare_seed": 123,

    "use_ddim_for_manual_tests": True,
    "manual_test_ddim_steps": 10,
    "manual_test_ddim_eta": 0.0,
    "eval_batch_size": 8,
    "eval_max_samples_per_task": 16,

    "compute_psnr": True,
    "compute_ssim": True,
    "eval_seed_base": 42,

    "output_path_stage1": "/kaggle/working/ddpm_merged_daretie_stage1_best.pth",
    "output_path_final": "/kaggle/working/ddpm_merged_daretie_final.pth",
    "results_json_path": "/kaggle/working/grid_search_results_daretie.json",
}

# =========================================================================
# UTILITY
# =========================================================================
def load_unet_state_dict(path: str, key: str = None):
    obj = torch.load(path, map_location="cpu")

    if key is not None and isinstance(obj, dict) and key in obj:
        sd = obj[key]
    else:
        sd = obj

    if any(k.startswith("_orig_mod.") for k in sd.keys()):
        sd = {k[len("_orig_mod."):]: v for k, v in sd.items()}
    if any(k.startswith("network.") for k in sd.keys()):
        sd = {k[len("network."):]: v for k, v in sd.items() if k.startswith("network.")}
    return OrderedDict(sd)

def compute_task_vector(theta_t, theta_0):
    tau = OrderedDict()
    for k, v0 in theta_0.items():
        if torch.is_floating_point(v0):
            tau[k] = theta_t[k].float() - v0.float()
        else:
            tau[k] = torch.zeros_like(v0)
    return tau

# --------------------------- DARE ---------------------------------------
def apply_dare_to_task_vectors(task_vectors: dict, drop_rate: float, seed: int):
    """
    DARE (Drop And REscale): per ogni task vector, azzera casualmente una
    frazione 'drop_rate' dei parametri e riscala i sopravvissuti per
    1/(1 - drop_rate).
    """
    rescale = 1.0 / (1.0 - drop_rate) if drop_rate < 1.0 else 0.0
    generator = torch.Generator(device="cpu").manual_seed(seed)

    dared = OrderedDict()
    for task_name, tau in task_vectors.items():
        dared_tau = OrderedDict()
        for k, v in tau.items():
            if torch.is_floating_point(v) and v.numel() > 0:
                keep_mask = (torch.rand(v.shape, generator=generator) > drop_rate).to(v.dtype)
                dared_tau[k] = v * keep_mask * rescale
            else:
                dared_tau[k] = v.clone()
        dared[task_name] = dared_tau
    return dared

# --------------------------- TIES ----------------------------------------
def elect_sign_per_parameter(task_vectors: dict, lambdas: dict):
    keys = next(iter(task_vectors.values())).keys()
    elected = {}
    for k in keys:
        acc = None
        for task_name, tau in task_vectors.items():
            v = tau[k]
            if not torch.is_floating_point(v):
                continue
            contrib = lambdas[task_name] * v
            acc = contrib.clone() if acc is None else acc + contrib
        if acc is None:
            elected[k] = None
        else:
            sign = torch.sign(acc)
            sign[sign == 0] = 1.0
            elected[k] = sign
    return elected

def merge_dare_ties(theta_0, task_vectors: dict, lambdas: dict):
    elected_signs = elect_sign_per_parameter(task_vectors, lambdas)

    merged = OrderedDict()
    for k, v0 in theta_0.items():
        if torch.is_floating_point(v0):
            acc = v0.float().clone()
            sign_k = elected_signs.get(k)
            for task_name, tau in task_vectors.items():
                tv = tau[k]
                if sign_k is not None:
                    agree_mask = (torch.sign(tv) == sign_k) | (tv == 0)
                    tv = tv * agree_mask.to(tv.dtype)
                acc = acc + lambdas[task_name] * tv
            merged[k] = acc.to(v0.dtype)
        else:
            merged[k] = v0.clone()
    return merged

# =========================================================================
# METRICHE
# =========================================================================
def _to_01(img):
    return (img.clamp(-1.0, 1.0) + 1.0) * 0.5

def psnr_per_sample(pred, target, eps=1e-10):
    p, t = _to_01(pred), _to_01(target)
    mse01 = F.mse_loss(p, t, reduction='none').mean(dim=[1, 2, 3])
    return 10.0 * torch.log10(1.0 / (mse01 + eps))

@torch.no_grad()
def compute_deterministic_val_loss(model, condition, target, n_steps=200, num_t=10):
    device = condition.device
    b = condition.shape[0]
    timesteps = torch.linspace(0, n_steps - 1, num_t, dtype=torch.long, device=device)
    fixed_noise = torch.randn_like(target)
    total_loss = 0.0

    for t_val in timesteps:
        t = torch.full((b,), t_val, device=device, dtype=torch.long)
        x_t = model.forward_diffusion(target, t, fixed_noise)
        sqrt_alpha_bar_t = model.sqrt_alpha_bar[t].view(-1, 1, 1, 1)
        sqrt_one_minus_alpha_bar_t = model.sqrt_one_minus_alpha_bar[t].view(-1, 1, 1, 1)

        v_target = sqrt_alpha_bar_t * fixed_noise - sqrt_one_minus_alpha_bar_t * target
        predicted_v = model.network(x_t, t.float(), condition)
        loss_t = F.mse_loss(predicted_v, v_target, reduction='none').mean(dim=[1, 2, 3])
        total_loss += loss_t
    return (total_loss / num_t).mean()


def compute_z_score_ranking(all_results):
    if not all_results: return all_results

    folders = list(all_results[0]["breakdown"].keys())
    stats = {}

    for f in folders:
        mses = [r["breakdown"][f]["mse"] for r in all_results if r["breakdown"][f]["mse"] is not None]
        lpips_vals = [r["breakdown"][f]["lpips"] for r in all_results if r["breakdown"][f]["lpips"] is not None]

        stats[f] = {
            "mu_mse": np.mean(mses) if mses else 0.0,
            "std_mse": (np.std(mses) + 1e-8) if mses else 1.0,
            "mu_lpips": np.mean(lpips_vals) if lpips_vals else 0.0,
            "std_lpips": (np.std(lpips_vals) + 1e-8) if lpips_vals else 1.0
        }

    for res in all_results:
        b = res["breakdown"]
        z_scores = []
        for f in folders:
            if "SU" in f and b[f]["lpips"] is not None:
                z = (b[f]["lpips"] - stats[f]["mu_lpips"]) / stats[f]["std_lpips"]
                z_scores.append(z)
            if ("SR" in f or "IR" in f) and b[f]["mse"] is not None:
                z = (b[f]["mse"] - stats[f]["mu_mse"]) / stats[f]["std_mse"]
                z_scores.append(z)
        res["combined_z_score"] = np.mean(z_scores) if z_scores else 0.0

    all_results.sort(key=lambda x: x["combined_z_score"])
    return all_results

# =========================================================================
# DATASET
# =========================================================================

class DegradedImageDataset(Dataset):
    def __init__(self, root: str, subsample_pct: float = 0.1, normalize_to_minus1_1: bool = True, seed: int = 42):
        self.normalize = normalize_to_minus1_1
        self.samples = []
        input_dir = os.path.join(root, "input", "npy")
        target_dir = os.path.join(root, "target", "npy")

        all_files = sorted(os.path.basename(p) for p in glob.glob(os.path.join(input_dir, "*.npy")))
        for f in all_files:
            in_path = os.path.join(input_dir, f)
            tgt_path = os.path.join(target_dir, f)
            if os.path.exists(tgt_path):
                self.samples.append((in_path, tgt_path))

        self.samples.sort()
        rng = random.Random(seed)
        num_samples = max(1, int(len(self.samples) * subsample_pct))
        self.samples = rng.sample(self.samples, num_samples)
        self.samples.sort()

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        in_path, tgt_path = self.samples[idx]
        x = torch.from_numpy(np.load(in_path).astype(np.float32))
        y = torch.from_numpy(np.load(tgt_path).astype(np.float32))
        if x.ndim == 3 and x.shape[-1] in (1, 3): x = x.permute(2, 0, 1)
        if y.ndim == 3 and y.shape[-1] in (1, 3): y = y.permute(2, 0, 1)
        if self.normalize:
            x = (x * 2.0) - 1.0
            y = (y * 2.0) - 1.0
        return x, y

def build_folder_loaders(dataset_base: str, folders: list, subsample_pct: float, batch_size: int, accelerator: Accelerator):
    prepared = OrderedDict()
    for folder in folders:
        root = os.path.join(dataset_base, folder)
        ds = DegradedImageDataset(root=root, subsample_pct=subsample_pct)
        loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
        prepared[folder] = accelerator.prepare(loader)
    return prepared

# =========================================================================
# BUILD MODEL / EVALUATE
# =========================================================================
    
def build_model_fn(state_dict, config=CONFIG):
    unet = ImprovedUNet(in_channels=config["in_channels"], cond_channels=config["cond_channels"], base_channels=config["base_channels"])
    unet.load_state_dict(state_dict)
    unet.eval()
    ddpm = DDPMvPrediction(unet, n_steps=config["n_steps"])
    ddpm.eval()
    return ddpm

@torch.no_grad()
def evaluate_per_folder(model, prepared_loaders, accelerator, config=CONFIG, folders_subset=None):
    use_ddim = config["use_ddim_for_manual_tests"]
    ddim_steps = config["manual_test_ddim_steps"]
    ddim_eta = config["manual_test_ddim_eta"]
    max_n = config["eval_max_samples_per_task"]
    seed_base = config["eval_seed_base"]

    unwrapped = accelerator.unwrap_model(model)
    ordered_names = list(prepared_loaders.keys())

    global lpips_metric
    lpips_metric = lpips_metric.to(accelerator.device).eval()

    breakdown = OrderedDict()
    for folder_idx, folder in enumerate(ordered_names):
        if folders_subset is not None and folder not in folders_subset: continue
        set_seed(seed_base + folder_idx)

        loader = prepared_loaders[folder]
        mses, psnrs, lpips_vals, val_losses = [], [], [], []

        for x, y in loader:
            val_loss = compute_deterministic_val_loss(unwrapped, x, y, n_steps=config["n_steps"], num_t=10)
            val_losses.append(val_loss.item())

            pred = unwrapped.sample_ddim(condition=x, ddim_steps=ddim_steps, eta=ddim_eta)

            mse_s = F.mse_loss(pred, y, reduction='none').mean(dim=[1, 2, 3])
            mses.extend(accelerator.gather_for_metrics(mse_s).cpu().tolist())

            if config["compute_psnr"]:
                psnrs.extend(accelerator.gather_for_metrics(psnr_per_sample(pred, y)).cpu().tolist())

            if "SU" in folder:
                l_batch = lpips_metric(pred, y).squeeze()
                if l_batch.ndim == 0: l_batch = l_batch.unsqueeze(0)
                lpips_vals.extend(accelerator.gather_for_metrics(l_batch).cpu().tolist())

            if max_n is not None and len(mses) >= max_n:
                break

        avg_val_loss = np.mean(val_losses) if val_losses else float("nan")
        avg_mse = np.mean(mses) if mses else float("nan")
        avg_psnr = np.mean(psnrs) if psnrs else None
        avg_lpips = np.mean(lpips_vals) if lpips_vals else None

        breakdown[folder] = {
            "val_loss_det": avg_val_loss,
            "mse": avg_mse,
            "psnr": avg_psnr,
            "lpips": avg_lpips,
            "n": len(mses)
        }

        msg = f"    [{folder:10s}] DetValLoss={avg_val_loss:.5f} | MSE={avg_mse:.5f}"
        if avg_lpips is not None: msg += f" | LPIPS={avg_lpips:.4f}"
        accelerator.print(msg)

    return breakdown

# =========================================================================
# MAIN GRID SEARCH
# =========================================================================
# La grid search è a due stadi: nello Stadio 1 si esplorano tutte le combinazioni 
# su dataset "puri" con pochi campioni. Nello Stadio 2 solo le migliori
# combinazioni vengono ritestate su dataset misti

def generate_full_grid(task_names, grid_values, drop_rates):
    """
    Genera tutte le combinazioni per drop_rate e valori di lambda.
    Ritorna una tupla: (drop_rate, lambda_dict)
    """
    lists_of_values = [grid_values[name] for name in task_names]
    lambda_combos = list(itertools.product(*lists_of_values))
    
    for dr in drop_rates:
        for combo in lambda_combos:
            yield dr, dict(zip(task_names, combo))

def run_lambda_search(theta_0, dared_task_vectors_dict, full_combos, prepared_loaders, accelerator, config=CONFIG, stage_label=""):
    total = len(full_combos)
    accelerator.print(f"\n[{stage_label}] {total} combinazioni (Drop Rate + Lambdas) da testare (DARE-TIES).")

    results = []
    for i, (drop_rate, lambdas) in enumerate(full_combos, start=1):
        # Seleziona il task vector corrispondente a questo drop_rate specifico
        dared_task_vectors = dared_task_vectors_dict[drop_rate]
        
        merged_sd = merge_dare_ties(theta_0, dared_task_vectors, lambdas)
        model = build_model_fn(merged_sd, config=config).to(accelerator.device)

        lam_str = ", ".join(f"{k}={v:.2f}" for k, v in lambdas.items())
        accelerator.print(f" --- [{stage_label} {i}/{total}] DropRate={drop_rate} | {lam_str}")

        bd = evaluate_per_folder(model, prepared_loaders, accelerator, config=config)

        results.append({
            "drop_rate": drop_rate,
            "lambda": lambdas,
            "breakdown": bd
        })
        del model
        accelerator.free_memory()
        torch.cuda.empty_cache()

    return results

def main():
    accelerator = Accelerator()
    accelerator.print("Avvio Grid Search Accelerate (DARE-TIES)...")

    theta0_sd = load_unet_state_dict(CONFIG["theta0_path"], CONFIG["ckpt_key"])
    task_vectors = {}
    for task_name, path in CONFIG["checkpoints"].items():
        sd = load_unet_state_dict(path, CONFIG["ckpt_key"])
        task_vectors[task_name] = compute_task_vector(sd, theta0_sd)

    # PRECALCOLO DARE per ogni possibile drop_rate per risparmiare risorse
    dared_task_vectors_dict = {}
    for dr in CONFIG["dare_drop_rate_values"]:
        dared_task_vectors_dict[dr] = apply_dare_to_task_vectors(
            task_vectors,
            drop_rate=dr,
            seed=CONFIG["dare_seed"],
        )

    stage1_folders = list(CONFIG["stage1_folder_to_task"].keys())
    stage1_loaders = build_folder_loaders(CONFIG["dataset_base"], stage1_folders, CONFIG["stage1_subsample_pct"], CONFIG["eval_batch_size"], accelerator)

    task_names = list(CONFIG["lambda_grid_values"].keys())
    all_combos = list(generate_full_grid(task_names, CONFIG["lambda_grid_values"], CONFIG["dare_drop_rate_values"]))

    # ------------------ STADIO 1 ------------------
    raw_results = run_lambda_search(theta0_sd, dared_task_vectors_dict, all_combos, stage1_loaders, accelerator, stage_label="STADIO 1")
    ranked_results = compute_z_score_ranking(raw_results)

    safe_zone = ranked_results[:CONFIG["stage1_safe_zone_size"]]

    accelerator.print("\n=== TOP COMBINAZIONI (STADIO 1) ===")
    for r in safe_zone:
        accelerator.print(f"Z-Score={r['combined_z_score']:.4f} -> DropRate={r['drop_rate']} | Lambda={r['lambda']}")

    # SALVATAGGIO STADIO 1
    if accelerator.is_main_process:
        best_stage1 = safe_zone[0]
        best_stage1_sd = merge_dare_ties(theta0_sd, dared_task_vectors_dict[best_stage1["drop_rate"]], best_stage1["lambda"])
        torch.save(best_stage1_sd, CONFIG["output_path_stage1"])
        print(f"\nSalvataggio modello vincitore STADIO 1 in {CONFIG['output_path_stage1']} completato.")

    # ------------------ STADIO 2 ------------------
    stage2_loaders = build_folder_loaders(CONFIG["dataset_base"], CONFIG["stage2_folders"], CONFIG["stage2_subsample_pct"], CONFIG["eval_batch_size"], accelerator)
    
    # Raccogliamo la lista (drop_rate, lambdas) dallo stage 1 per usarla nello stage 2
    safe_combos = [(r["drop_rate"], r["lambda"]) for r in safe_zone]

    final_raw_results = run_lambda_search(theta0_sd, dared_task_vectors_dict, safe_combos, stage2_loaders, accelerator, stage_label="STADIO 2")
    final_ranked_results = compute_z_score_ranking(final_raw_results)

    best_final = final_ranked_results[0]
    accelerator.print("\n=== MIGLIORE IN ASSOLUTO (STADIO 2) ===")
    accelerator.print(f"Z-Score  = {best_final['combined_z_score']:.4f}")
    accelerator.print(f"DropRate = {best_final['drop_rate']}")
    accelerator.print(f"Lambda   = {best_final['lambda']}")

    # SALVATAGGIO FINALE STADIO 2
    if accelerator.is_main_process:
        with open(CONFIG["results_json_path"], "w") as f:
            json.dump(final_ranked_results, f, indent=2)

        best_merged_sd = merge_dare_ties(theta0_sd, dared_task_vectors_dict[best_final["drop_rate"]], best_final["lambda"])
        torch.save(best_merged_sd, CONFIG["output_path_final"])
        print(f"Salvataggio modello vincitore FINALE completato in {CONFIG['output_path_final']}.")

if __name__ == "__main__":
    main()

In [ ]:
!accelerate launch --num_processes=2 /kaggle/working/train_ddp_accelerate.py

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import os

from train_ddp_accelerate import ImprovedUNet, DDPMvPrediction, DegradedImageDataset

# =========================================================================
# CONFIGURAZIONE, CARICAMENTO MODELLO E DATASET
# =========================================================================

MODEL_PATH = "/kaggle/input/datasets/phantasm04/pesi-ties-lpips/ddpm_merged_daretie_final.pth"

# ATTENZIONE: Scegli la sottocartella specifica del task combinato che vuoi testare
# (es. la cartella contenente "star_removal_image_restoration" o simili)
DATASET_DIR = "/kaggle/input/datasets/phantasm04/merged/dataset_merged/SR_SU" 

INDICES_TO_VISUALIZE = [1, 2, 7, 11, 15, 19]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Caricamento del modello fuso in corso...")
unet = ImprovedUNet(in_channels=3, cond_channels=3, base_channels=64)
unet.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
unet.to(DEVICE)
unet.eval()

ddpm = DDPMvPrediction(unet, n_steps=200).to(DEVICE)
ddpm.eval()

print(f"Caricamento dataset da: {DATASET_DIR}")
dataset = DegradedImageDataset(DATASET_DIR, normalize_to_minus1_1=True)

def denormalize(t):
    """Riporta i tensori dal range [-1, 1] a [0, 1] per matplotlib."""
    t = (t + 1.0) / 2.0
    return torch.clamp(t, 0.0, 1.0)

# =========================================================================
# INFERENZA E PLOT
# =========================================================================

n_images = len(INDICES_TO_VISUALIZE)
fig, axes = plt.subplots(n_images, 3, figsize=(15, 5 * n_images))

# Gestione per il caso in cui ci sia un solo indice
if n_images == 1:
    axes = [axes]

print("Inizio inferenza e generazione immagini...")
with torch.no_grad():
    for i, idx in enumerate(INDICES_TO_VISUALIZE):
        if idx >= len(dataset):
            print(f"[Warning] L'indice {idx} supera la dimensione del dataset ({len(dataset)}). Salto.")
            continue
            
        cond, target = dataset[idx]
        
        cond_batch = cond.unsqueeze(0).to(DEVICE)
        
        # Generiamo la predizione usando DDIM (molto più veloce per visualizzare)
        print(f"Campionamento per l'indice {idx} (DDIM)...")
        pred = ddpm.sample_ddim(condition=cond_batch, ddim_steps=25, eta=0.0)
        
        # Prepariamo i tensori per il plotting: (C, H, W) -> (H, W, C)
        cond_img = denormalize(cond).cpu().permute(1, 2, 0).numpy()
        target_img = denormalize(target).cpu().permute(1, 2, 0).numpy()
        pred_img = denormalize(pred.squeeze(0)).cpu().permute(1, 2, 0).numpy()
        
        # Plot Input (Condition)
        axes[i, 0].imshow(cond_img)
        axes[i, 0].set_title(f"Input Degradato (Idx: {idx})", fontsize=12)
        axes[i, 0].axis('off')
        
        # Plot Target
        axes[i, 1].imshow(target_img)
        axes[i, 1].set_title(f"Target Ground Truth (Idx: {idx})", fontsize=12)
        axes[i, 1].axis('off')
        
        # Plot Prediction
        axes[i, 2].imshow(pred_img)
        axes[i, 2].set_title(f"Prediction (Modello Fuso)", fontsize=12)
        axes[i, 2].axis('off')

plt.tight_layout()
plt.show()